# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kavithamuddagowni3-cyber/flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
## 1. Build the feature vector

The feature vector converts the raw content dataset into a machine-learning-ready format.

The process includes:

- Selecting observable signals that are available before prediction.
- Creating engineered features such as logarithmic transformations for high-volume metrics.
- Handling missing values.
- Encoding categorical variables into numerical values.
- Separating features from the target label.

The final feature vector contains search visibility, engagement, content quality, and freshness signals used by the model to rank content refresh opportunities.

The target variable is:

`is_declining_label`

The model should learn patterns from the available signals without using product decision outputs or private information.

import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

# Load raw dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Original shape:", df.shape)


# -----------------------------
# Create engineered features
# -----------------------------

# Log transform high-volume metrics
for col in ["impressions_90d", "clicks_90d", "sessions_90d"]:
    if col in df.columns:
        df[f"log_{col}"] = np.log1p(df[col])


# Create label
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)


# -----------------------------
# Select numerical features
# -----------------------------

numeric_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "word_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d"
]

numeric_features = [
    col for col in numeric_features
    if col in df.columns
]


# Fill missing numerical values
df[numeric_features] = (
    df[numeric_features]
    .fillna(0)
)


# -----------------------------
# Handle categorical features
# -----------------------------

categorical_features = [
    "content_type",
    "intent",
    "competition_level"
]

categorical_features = [
    col for col in categorical_features
    if col in df.columns
]


df[categorical_features] = (
    df[categorical_features]
    .fillna("unknown")
)


# One-hot encode categories
if categorical_features:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )

    encoded = encoder.fit_transform(
        df[categorical_features]
    )

    encoded_df = pd.DataFrame(
        encoded,
        columns=encoder.get_feature_names_out(
            categorical_features
        )
    )

else:
    encoded_df = pd.DataFrame()


# Combine feature vector
feature_vector = pd.concat(
    [
        df[numeric_features],
        encoded_df,
        df["is_declining_label"]
    ],
    axis=1
)


print("Feature vector shape:", feature_vector.shape)

feature_vector.head()

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
## 2. Feature notes (meaning, missing, categorical, available-when?)

The selected features are observable signals that are available before making a content review decision.

| Feature | Meaning | Missing value handling | Type | Available when? |
|---|---|---|---|---|
| impressions_90d | Number of search impressions received in the last 90 days | Filled with 0 | Numeric | Before prediction |
| clicks_90d | Number of search clicks in the last 90 days | Filled with 0 | Numeric | Before prediction |
| sessions_90d | Number of user sessions in the last 90 days | Filled with 0 | Numeric | Before prediction |
| ctr | Click-through rate from search results | Filled with 0 | Numeric | Before prediction |
| avg_position | Average search ranking position | Filled with 0 | Numeric | Before prediction |
| content_age_days | Age of the content page in days | Filled with 0 | Numeric | Before prediction |
| word_count | Approximate content length indicator | Filled with 0 | Numeric | Before prediction |
| engagement_rate | User engagement measurement | Filled with 0 | Numeric | Before prediction |
| scroll_rate | User scrolling behavior signal | Filled with 0 | Numeric | Before prediction |
| log_impressions_90d | Log-transformed impressions to reduce scale differences | Calculated after filling missing values | Numeric engineered feature | Before prediction |
| log_clicks_90d | Log-transformed clicks | Calculated after filling missing values | Numeric engineered feature | Before prediction |
| log_sessions_90d | Log-transformed sessions | Calculated after filling missing values | Numeric engineered feature | Before prediction |
| content_type | Type/category of content | Filled with "unknown" | Categorical | Before prediction |
| intent | Search/content intent category | Filled with "unknown" | Categorical | Before prediction |
| competition_level | Competition category associated with content | Filled with "unknown" | Categorical | Before prediction |

The label (`is_declining_label`) is created separately from `trend_direction` and is not used as an input feature.

All selected features represent information available before the ranking decision. Future information is excluded to reduce leakage risk.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
## 3. The leakage hunt

A leakage check was performed to make sure the model does not use information that would not be available at prediction time.

Potential leakage risks considered:

### 1. Label-derived columns
The target variable is:

`is_declining_label`

created from:

`trend_direction == "down"`

The label itself is excluded from the feature set. No columns directly created from the label are used as model inputs.

### 2. Future information
The model uses historical 90-day performance signals only.

Features such as impressions, clicks, sessions, CTR, position, and engagement are measured before the prediction decision. Future outcomes are not included in the feature vector.

### 3. Product decision flags
Product-generated fields such as:
- `health_score`
- `priority_score`
- `action_type`

are not included because they represent existing decisions. Using them would cause the model to copy FlyRank's rules instead of learning new patterns.

The leakage test confirms that the model relies only on observable signals available before prediction.

import pandas as pd

# Load feature vector
df = pd.read_csv("../../data/processed/refresh_feature_vector.csv")


print("===== LABEL LEAKAGE CHECK =====")

# Check if label is accidentally used as a feature
feature_columns = [
    col for col in df.columns 
    if col != "is_declining_label"
]

print("Label column in features:",
      "is_declining_label" in feature_columns)


print("\n===== PRODUCT FLAG CHECK =====")

product_flags = [
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier"
]

for col in product_flags:
    print(col, "exists:",
          col in feature_columns)


print("\n===== SUSPICIOUS COLUMN CHECK =====")

keywords = [
    "label",
    "target",
    "future",
    "next",
    "outcome",
    "priority",
    "score"
]

suspicious = [
    col for col in feature_columns
    if any(word in col.lower() for word in keywords)
]

print("Potentially suspicious features:")
print(suspicious)

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
## 4. What I excluded and why

Some fields were intentionally excluded from the model because they could introduce leakage, privacy risks, or cause the model to copy existing decisions instead of learning from observable signals.

| Excluded field | Why excluded |
|---|---|
| `content_id` | Used only as an identifier; it has no meaningful predictive information. |
| `client_id` | Anonymized identifier used for grouping/validation, not as a prediction feature because the model should learn content patterns, not client identity. |
| Raw URLs | Excluded because they may reveal private website information and do not represent generalizable signals. |
| Raw queries/keywords | Excluded because they contain sensitive search information and may expose private user behavior. |
| Client names/domains | Excluded to protect confidentiality and prevent identifying organizations. |
| `health_score` | Excluded because it is a product-generated decision score and using it would make the model copy existing rules. |
| `priority_score` | Excluded because it is already an output from a decision system, not an independent observable signal. |
| `action_type` | Excluded because it represents the recommended action and would leak the expected output. |
| `refresh_tier` | Excluded because it is a rule-based categorization that may encode the same decision we want to learn. |
| `is_declining_label` | Excluded from features because it is the target variable being predicted. |
| Future performance metrics | Excluded because they would not be available at prediction time and would create future leakage. |

The final model uses only observable features available before the decision point. These exclusions help ensure the model learns meaningful content performance patterns instead of memorizing existing decisions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.